# Chapter 3 — Gender and Market Segmentation: Visualisation DraftThis notebook renders 19 table datasets (including 36 uniquely numbered gender- and segment-specific fragrance-example exhibits) and 6 figures for Chapter 3 from the validated exports produced by Notebook 18. Run Notebook 18 through its export cell first.This is a first visual draft. The existing Notebook 14 style is reused provisionally; table design will be reviewed before a global redesign is applied to the whole book.**Methodological conventions:** dominant accord = position 1; dyad = positions 1+2; triad = positions 1+2+3, preserving source order. One profile per fragrance at each applicable level. Minimum support = 50 fragrances per classification–profile. Median segmentation is independent within each classification and analytical level. Votes are platform activity, not sales or unique consumers; ratings are platform evaluations, not objective quality measures.

## 0. Setup, load validated exports and prepare all tables

In [0]:
# ============================================================# CHAPTER 3 VISUALISATION DRAFT — SETUP, DATA AND TABLE RENDERER# ============================================================import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom matplotlib.colors import LinearSegmentedColormapfrom pathlib import Pathimport textwrapfrom io import BytesIOimport base64DATA_DIR = Path("/Volumes/main/fragrance_analytics/reference_data/chapter_3")OUTPUT_DIR = Path("/Volumes/main/fragrance_analytics/reference_data/visualizations")OUTPUT_DIR.mkdir(parents=True, exist_ok=True)GENDERS = ["Men", "Women", "Unisex"]LEVELS = ["Dominant accords", "Dyads", "Triads"]SEGMENTS = [    "Standout Profiles",    "Hidden Favourites",    "Widely Explored",    "Quiet Profiles"]# Shared book palette, aligned with the Matplotlib table design in Notebook 15.BOOK_STYLE = {    "background": "#F7F3EC",    "surface": "#F7F3EC",    "text": "#302824",    "muted": "#756B64",    "accent": "#A47C59",    "accent_light": "#D8C3AD",    "rule": "#E5DDD2",    "grid": "#E9E2D8",    "men": "#687F8C",    "women": "#B47C83",    "unisex": "#A58B68",    "other": "#B9B2AC"}# Keep the existing figure code unchanged while sharing one palette.STYLE = {    "bg": BOOK_STYLE["background"],    "text": BOOK_STYLE["text"],    "muted": BOOK_STYLE["muted"],    "accent": BOOK_STYLE["accent"],    "grid": BOOK_STYLE["grid"],    "Men": BOOK_STYLE["men"],    "Women": BOOK_STYLE["women"],    "Unisex": BOOK_STYLE["unisex"]}BOOK_TYPOGRAPHY = {    "display": "DejaVu Serif",    "body": "DejaVu Sans"}SEGMENT_COLORS = {    "Standout Profiles": "#A47C59",    "Hidden Favourites": "#687F8C",    "Widely Explored": "#B47C83",    "Quiet Profiles": "#A58B68"}plt.rcParams.update({    "figure.facecolor": BOOK_STYLE["background"],    "axes.facecolor": BOOK_STYLE["background"],    "savefig.facecolor": BOOK_STYLE["background"],    "figure.edgecolor": BOOK_STYLE["background"],    "font.family": BOOK_TYPOGRAPHY["body"],    "text.color": BOOK_STYLE["text"],    "axes.labelcolor": BOOK_STYLE["muted"],    "xtick.color": BOOK_STYLE["text"],    "ytick.color": BOOK_STYLE["text"],    "savefig.dpi": 300})# ============================================================# 1. LOAD CHAPTER 3 DATA# ============================================================def load_csv(name):    path = DATA_DIR / name    if not path.exists():        raise FileNotFoundError(            f"{path} is missing. Run Notebook 18 through its export cell."        )    return pd.read_csv(path)gender_dist = load_csv("chapter_3_gender_distribution.csv")prevalence = load_csv("chapter_3_accord_prevalence.csv")thresholds = load_csv("chapter_3_gender_thresholds.csv")segment_summary = load_csv("chapter_3_gender_segment_summary.csv")dominant = load_csv("chapter_3_dominant_accord_profiles.csv")dyads = load_csv("chapter_3_dyad_profiles.csv")triads = load_csv("chapter_3_triad_profiles.csv")examples = load_csv("chapter_3_top_three_fragrance_examples.csv")level_comparison = load_csv("chapter_3_gender_level_comparison.csv")segment_comparison = load_csv("chapter_3_gender_segment_comparison.csv")# ============================================================# 2. PREPARE PROFILE FREQUENCIES AND EXAMPLES# ============================================================PROFILE_DATA = {    "Dominant accords": dominant,    "Dyads": dyads,    "Triads": triads}freq_parts = []for level, pdf in PROFILE_DATA.items():    entity = {        "Dominant accords": "dominant_accord",        "Dyads": "dyad",        "Triads": "triad"    }[level]    part = pdf[        ["gender_clean", entity, "fragrance_count"]    ].copy()    part.columns = [        "classification",        "profile",        "profile_frequency"    ]    part["analytical_level"] = level    freq_parts.append(part)profile_freq = pd.concat(freq_parts, ignore_index=True)examples["profile"] = (    examples["dominant_accord"]    .fillna(examples["dyad"])    .fillna(examples["triad"]))examples = examples.merge(    profile_freq,    on=["analytical_level", "classification", "profile"],    how="left")# ============================================================# 3. VALIDATE SOURCE DATA# ============================================================assert (    len(gender_dist) == 3    and set(gender_dist.gender_clean) == set(GENDERS)), "Catalogue distribution must contain Men, Women and Unisex."assert (    set(thresholds.minimum_fragrance_support.astype(int)) == {50}), "Unexpected minimum support threshold."assert (    len(dominant),    len(dyads),    len(triads)) == (95, 276, 104), "Unexpected eligible profile counts."assert examples.profile_frequency.notna().all(), (    "Example data is incomplete or profile frequencies are missing.")# Every eligible classification–profile group must have at least three# examples. More than three rows are valid when the third position is tied.example_group_cols = ["analytical_level", "classification", "profile"]example_group_sizes = examples.groupby(example_group_cols).size()expected_profile_counts = {    "Dominant accords": 95,    "Dyads": 276,    "Triads": 104}assert example_group_sizes.groupby(level=0).size().to_dict() == expected_profile_counts, (    "The example export does not cover every eligible profile.")assert example_group_sizes.ge(3).all(), (    "At least three fragrance examples are required for every eligible profile.")# The source export must retain all records ranked within the top three# positions. A tie at rank 3 may therefore produce more than three rows.assert "rank_by_votes" in examples.columns, (    "The example export must include rank_by_votes for validation and ordering.")examples["rank_by_votes"] = pd.to_numeric(    examples["rank_by_votes"], errors="raise")assert examples["rank_by_votes"].between(1, 3).all(), (    "The example export contains a rank outside the top three positions.")assert examples.groupby(example_group_cols)["rank_by_votes"].min().eq(1).all(), (    "Each eligible profile must include its highest-vote fragrance.")assert set(examples["classification"]) == set(GENDERS), (    "Fragrance examples must include Men, Women and Unisex classifications.")print(    "Source data loaded and validation checks passed.",    f"({len(examples):,} fragrance examples across {len(example_group_sizes):,} eligible profiles.)")# ============================================================# 4. TABLE VALUE FORMATTING# ============================================================def _format_table_value(value):    """Format display values without changing the underlying DataFrame."""    if pd.isna(value):        return ""    if isinstance(value, (int, np.integer)) and not isinstance(        value, (bool, np.bool_)    ):        return f"{value:,}"    if isinstance(value, (float, np.floating)):        if not np.isfinite(value):            return ""        return f"{value:,.2f}"    return str(value)# ============================================================# 5. PUBLICATION TABLE RENDERER# ============================================================def render_book_table(    data,    title,    subtitle,    table_number,    compact_title_gap=False):    """Render a publication table using the established Notebook 15 design."""    display_data = data.copy()    n_rows, n_cols = display_data.shape    raw_widths = []    for column in display_data.columns:        samples = (            [str(column)]            + display_data[column].fillna("").astype(str).tolist()        )        max_chars = max(            (len(value) for value in samples),            default=8        )        raw_widths.append(max(8, min(30, max_chars + 2)))    width_total = sum(raw_widths)    column_widths = [        value / width_total        for value in raw_widths    ]    figure_width = max(        10.5,        min(24, 2.0 + n_cols * 1.55)    )    figure_height = max(        3.7,        2.45 + n_rows * 0.56    )    fig, ax = plt.subplots(        figsize=(figure_width, figure_height),        dpi=180    )    fig.patch.set_facecolor(BOOK_STYLE["background"])    ax.set_facecolor(BOOK_STYLE["background"])    ax.set_position([0, 0, 1, 1])    ax.axis("off")    wrapped_headers = [        textwrap.fill(            str(label),            width=max(9, int(width * 105))        )        for label, width in zip(            display_data.columns,            column_widths        )    ]    cell_text = [        [            _format_table_value(value)            for value in row        ]        for row in display_data.itertuples(            index=False,            name=None        )    ]    table = ax.table(        cellText=cell_text,        colLabels=wrapped_headers,        colWidths=column_widths,        cellLoc="left",        colLoc="center",        bbox=[0.055, 0.025, 0.92, 0.76]    )    table.auto_set_font_size(False)    body_font_size = 12.5 if n_cols <= 7 else 10.5    header_font_size = 9.5 if n_cols <= 7 else 7.8    for (row, col), cell in table.get_celld().items():        cell.set_edgecolor(BOOK_STYLE["rule"])        cell.set_linewidth(0.65)        cell.PAD = 0.12        if row == 0:            cell.set_facecolor("#40332E")            cell.set_text_props(                color="#FFFDF9",                weight="bold",                fontsize=header_font_size,                fontfamily=BOOK_TYPOGRAPHY["body"],                ha="center",                va="center"            )            cell.set_height(                0.105 if n_cols <= 7 else 0.14            )        else:            cell.set_facecolor(                "#FFFDF9" if row % 2 else "#F0E9DF"            )            cell.set_text_props(                color=BOOK_STYLE["text"],                fontsize=body_font_size,                fontfamily=BOOK_TYPOGRAPHY["body"],                va="center"            )            if col == 0:                cell.set_text_props(                    color=BOOK_STYLE["text"],                    weight="bold",                    fontsize=body_font_size,                    fontfamily=BOOK_TYPOGRAPHY["body"],                    ha="left",                    va="center"                )            else:                cell.set_text_props(ha="right")            cell.set_height(0.085)    fig.text(        0.055,        0.955,        title,        ha="left",        va="top",        color=BOOK_STYLE["text"],        fontsize=20,        fontfamily=BOOK_TYPOGRAPHY["display"],        fontweight="bold"    )    if subtitle:        fig.text(            0.057,            0.895,            textwrap.fill(subtitle, width=145),            ha="left",            va="top",            color=BOOK_STYLE["muted"],            fontsize=10.5,            fontfamily=BOOK_TYPOGRAPHY["body"]        )    return fig# ============================================================# 6. PNG DISPLAY AND EXPORT# ============================================================def display_book_table_png(fig):    """Display a PNG in Databricks using a light, dark-mode-protected container."""    buffer = BytesIO()    fig.savefig(        buffer,        format="png",        dpi=180,        bbox_inches=None,        facecolor=BOOK_STYLE["background"],        edgecolor=BOOK_STYLE["background"]    )    buffer.seek(0)    image_b64 = base64.b64encode(        buffer.getvalue()    ).decode("ascii")    displayHTML(        '<div class="nodarkmode" '        'style="background:#F7F3EC;padding:0;margin:0;">'        f'<img src="data:image/png;base64,{image_b64}" '        'style="display:block;max-width:100%;height:auto;'        'background:#F7F3EC;">'        '</div>'    )def save_book_table_png(fig, table_number):    """Save a high-resolution PNG directly to the Chapter 3 visualisations folder."""    safe_number = str(table_number).replace(".", "_")    output_path = OUTPUT_DIR / f"table_{safe_number}.png"    # Save directly to the final destination.    fig.savefig(        str(output_path),        format="png",        dpi=300,        bbox_inches=None,        pad_inches=0,        facecolor=BOOK_STYLE["background"],        edgecolor=BOOK_STYLE["background"]    )    # Validate the actual destination file.    if not output_path.exists():        raise FileNotFoundError(            f"PNG export was not created: {output_path}"        )    file_size = output_path.stat().st_size    if file_size == 0:        raise IOError(            f"PNG export is empty: {output_path}"        )    print(        f"Saved PNG: {output_path} "        f"({file_size:,} bytes)"    )# ============================================================# 7. TABLE DISPLAY FUNCTION# ============================================================def show_table(    df,    number,    title,    subtitle,    classifications=None,    segments=None):    """    Render ordinary tables individually.    For example tables 3.7, 3.11 and 3.15, render    four segment tables per classification.    """    if number in ("3.7", "3.11", "3.15"):        classifications = (            GENDERS            if classifications is None            else classifications        )        segments = (            SEGMENTS            if segments is None            else segments        )        exhibit_suffixes = {            (classification, segment): chr(ord("a") + i)            for i, (classification, segment) in enumerate(                (                    (classification, segment)                    for classification in GENDERS                    for segment in SEGMENTS                )            )        }        for classification in classifications:            for segment in segments:                segment_data = (                    df[                        (df["Classification"] == classification)                        & (df["Segment"] == segment)                    ]                    .copy()                    .reset_index(drop=True)                )                if segment_data.empty:                    raise ValueError(                        f"No fragrance examples found for "                        f"{classification} — {segment} "                        f"in exhibit series {number}."                    )                # Classification and segment are stated in the title.                # Keep the accord/profile value on every fragrance row.                segment_data = segment_data.drop(                    columns=["Classification", "Segment"]                )                unique_number = (                    f"{number}"                    f"{exhibit_suffixes[(classification, segment)]}"                )                table_title = (                    f"Table {unique_number} — {title} — "                    f"{classification} — {segment}"                )                table_subtitle = (                    f"{subtitle} "                    f"Classification: {classification}. "                    f"Segment: {segment}."                )                print(                    f"Table {unique_number} — "                    f"{classification} — {segment}: "                    f"{len(segment_data):,} fragrance examples"                )                fig = render_book_table(                    segment_data,                    table_title,                    table_subtitle,                    unique_number,                    compact_title_gap=True                )                try:                    save_book_table_png(                        fig,                        unique_number                    )                    display_book_table_png(fig)                finally:                    plt.close(fig)        return    fig = render_book_table(        df,        title,        subtitle,        number    )    try:        save_book_table_png(fig, number)        display_book_table_png(fig)    finally:        plt.close(fig)# ============================================================# 8. PREPARE TABLE 3.1 — CATALOGUE DISTRIBUTION# ============================================================t31 = gender_dist.rename(    columns={        "gender_clean": "Classification",        "fragrance_count": "Fragrance count",        "catalogue_share_pct": "Catalogue share (%)"    })[    [        "Classification",        "Fragrance count",        "Catalogue share (%)"    ]]# ============================================================# 9. PREPARE TABLE 3.2 — MOST PREVALENT ACCORDS# ============================================================t32 = (    prevalence.sort_values(        ["gender_clean", "prevalence_pct", "accord"],        ascending=[True, False, True]    )    .groupby(        "gender_clean",        group_keys=False    )    .head(5)    .rename(        columns={            "gender_clean": "Classification",            "accord": "Accord",            "fragrance_count": "Fragrance count",            "prevalence_pct": "Prevalence (%)"        }    )[        [            "Classification",            "Accord",            "Fragrance count",            "Prevalence (%)"        ]    ])# ============================================================# 10. PREPARE TABLE 3.3 — PREVALENCE DIFFERENCES# ============================================================wide = (    prevalence.pivot(        index="accord",        columns="gender_clean",        values="prevalence_pct"    )    .reset_index())wide["Maximum spread (pp)"] = (    wide[GENDERS].max(axis=1)    - wide[GENDERS].min(axis=1))t33 = (    wide.nlargest(10, "Maximum spread (pp)")    .rename(columns={"accord": "Accord"}))[    [        "Accord",        "Men",        "Women",        "Unisex",        "Maximum spread (pp)"    ]]# ============================================================# 11. TABLE PREPARATION HELPERS# ============================================================def thresholds_table(level):    return thresholds[        thresholds.analytical_level == level    ].rename(        columns={            "classification": "Classification",            "eligible_profile_count": "Eligible profiles",            "minimum_fragrance_support": "Minimum support",            "attention_median": "Median attention (votes)",            "rating_median": "Median rating"        }    )[        [            "Classification",            "Eligible profiles",            "Minimum support",            "Median attention (votes)",            "Median rating"        ]    ]def counts_table(level):    d = segment_summary[        segment_summary.analytical_level == level    ]    return (        d.pivot_table(            index="segment",            columns="gender_clean",            values="profile_count",            aggfunc="sum",            fill_value=0        )        .reindex(SEGMENTS)        .reindex(columns=GENDERS)        .reset_index()        .rename(columns={"segment": "Segment"})    )def metrics_table(level):    d = segment_summary[        segment_summary.analytical_level == level    ].copy()    d["segment"] = pd.Categorical(        d.segment,        categories=SEGMENTS,        ordered=True    )    d["gender_clean"] = pd.Categorical(        d.gender_clean,        categories=GENDERS,        ordered=True    )    d = d.sort_values(        ["gender_clean", "segment"]    )    return d.rename(        columns={            "gender_clean": "Classification",            "segment": "Segment",            "profile_count": "Profiles",            "fragrance_assignments": "Fragrance assignments",            "mean_votes_per_profile": "Mean votes",            "mean_rating_per_profile": "Mean rating",            "median_fragrances_per_profile": "Median profile frequency"        }    )[        [            "Classification",            "Segment",            "Profiles",            "Fragrance assignments",            "Mean votes",            "Mean rating",            "Median profile frequency"        ]    ]def example_table(level):    # Keep vote rank for ordering, but do not expose rank numbers.    d = examples[        examples.analytical_level == level    ].sort_values(        [            "classification",            "segment",            "profile",            "rank_by_votes"        ],        ascending=[True, True, True, True]    )    return d.rename(        columns={            "classification": "Classification",            "segment": "Segment",            "profile": "Accord / profile",            "fragrance_name": "Fragrance",            "brand": "Brand",            "rating": "Rating",            "rating_count": "Votes",            "profile_frequency": "Profile frequency"        }    )[        [            "Classification",            "Segment",            "Accord / profile",            "Fragrance",            "Brand",            "Rating",            "Votes",            "Profile frequency"        ]    ]# ============================================================# 12. PREPARE TABLES 3.4–3.15# ============================================================t34 = thresholds_table("Dominant accords")t35 = counts_table("Dominant accords")t36 = metrics_table("Dominant accords")t37 = example_table("Dominant accords")t38 = thresholds_table("Dyads")t39 = counts_table("Dyads")t310 = metrics_table("Dyads")t311 = example_table("Dyads")t312 = thresholds_table("Triads")t313 = counts_table("Triads")t314 = metrics_table("Triads")t315 = example_table("Triads")# ============================================================# 13. PREPARE TABLES 3.16–3.19# ============================================================t316 = (    level_comparison.pivot(        index="gender_clean",        columns="analytical_level",        values="eligible_profile_count"    )    .reindex(        index=GENDERS,        columns=LEVELS    )    .reset_index()    .rename(        columns={            "gender_clean": "Classification",            "Dominant accords": "Dominant accord profiles",            "Dyads": "Dyad profiles",            "Triads": "Triad profiles"        }    ))t317 = (    level_comparison.pivot(        index="gender_clean",        columns="analytical_level",        values="fragrance_assignments"    )    .reindex(        index=GENDERS,        columns=LEVELS    )    .reset_index()    .rename(        columns={            "gender_clean": "Classification",            "Dominant accords": "Dominant accord assignments",            "Dyads": "Dyad assignments",            "Triads": "Triad assignments"        }    ))t318 = level_comparison.rename(    columns={        "gender_clean": "Classification",        "analytical_level": "Analytical level",        "eligible_profile_count": "Eligible profiles",        "mean_votes_per_profile": "Mean votes per profile",        "mean_rating_per_profile": "Mean rating per profile"    })[    [        "Classification",        "Analytical level",        "Eligible profiles",        "Mean votes per profile",        "Mean rating per profile"    ]]t319 = segment_comparison.rename(    columns={        "analytical_level": "Analytical level",        "segment": "Segment"    })# ============================================================# 14. REGISTER ALL CHAPTER 3 TABLES# ============================================================TABLES = [    (        "3.1", t31,        "Catalogue Distribution by Classification",        "Fragrance counts and shares within the classified catalogue."    ),    (        "3.2", t32,        "Most Prevalent Accords by Classification",        "Five most prevalent recorded accords within each classification."    ),    (        "3.3", t33,        "Accords with the Largest Prevalence Differences",        "Top ten accords ranked by maximum minus minimum prevalence, in percentage points."    ),    (        "3.4", t34,        "Dominant Accord Segmentation Benchmarks",        "Classification-specific medians and minimum support."    ),    (        "3.5", t35,        "Dominant Accord Profiles by Segment and Classification",        "Eligible dominant accord profile counts by relative segment."    ),    (        "3.6", t36,        "Dominant Accord Segment Metrics",        "Profile counts, assignments and unweighted profile-level metrics."    ),    (        "3.7", t37,        "Fragrance Examples for Dominant Accord Profiles",        "Complete top-three candidate review; final manuscript uses selected profiles."    ),    (        "3.8", t38,        "Dyad Segmentation Benchmarks",        "Classification-specific medians and minimum support."    ),    (        "3.9", t39,        "Dyad Profiles by Segment and Classification",        "Eligible dyad profile counts by relative segment."    ),    (        "3.10", t310,        "Dyad Segment Metrics",        "Profile counts, assignments and unweighted profile-level metrics."    ),    (        "3.11", t311,        "Fragrance Examples for Dyad Profiles",        "Complete top-three candidate review; final manuscript uses selected profiles."    ),    (        "3.12", t312,        "Triad Segmentation Benchmarks",        "Classification-specific medians and minimum support."    ),    (        "3.13", t313,        "Triad Profiles by Segment and Classification",        "Eligible triad profile counts by relative segment."    ),    (        "3.14", t314,        "Triad Segment Metrics",        "Profile counts, assignments and unweighted profile-level metrics."    ),    (        "3.15", t315,        "Fragrance Examples for Triad Profiles",        "Complete top-three candidate review; final manuscript uses selected profiles."    ),    (        "3.16", t316,        "Eligible Profiles Across Analytical Levels",        "Profiles meeting the support threshold by classification and level."    ),    (        "3.17", t317,        "Fragrance Assignments Across Analytical Levels",        "Assignments within eligible profiles; totals are not unique catalogue counts."    ),    (        "3.18", t318,        "Mean Attention and Rating Across Analytical Levels",        "Unweighted means across eligible profiles within each classification and level."    ),    (        "3.19", t319,        "Segment Distribution Across Analytical Levels",        "Eligible profile counts by segment, classification and analytical level."    )]print(    "Loaded source exports and prepared",    len(TABLES),    "table datasets.")

In [ ]:
# ============================================================# NOTEBOOK 19 — EXPORT TYPOGRAPHY-READY CHAPTER 3 DATA# ============================================================## This cell writes publication datasets as UTF-8 CSV files.# Copy the resulting chapter_3_typst folder into the Typst project.# Typst can then read each file with csv("path/to/file.csv").## The example tables contain every eligible profile and all fragrance# examples ranked within the top three vote positions. Tied third-place# records are retained when they are present in the Notebook 18 export.# The rank field is retained in the audit export but omitted from the# publication-facing example tables.from pathlib import Pathimport reTYPST_DATA_DIR = Path(    "/Volumes/main/fragrance_analytics/reference_data/typst/chapter_3")TYPST_DATA_DIR.mkdir(parents=True, exist_ok=True)def typst_slug(value):    value = str(value).strip().lower()    value = re.sub(r"[^a-z0-9]+", "_", value)    return value.strip("_")# Export all 19 publication table datasets exactly as prepared above.table_manifest = []for number, data, title, caption in TABLES:    file_name = f"table_{number.replace('.', '_')}_{typst_slug(title)}.csv"    file_path = TYPST_DATA_DIR / file_name    data.to_csv(file_path, index=False, encoding="utf-8")    table_manifest.append({        "table_number": number,        "title": title,        "caption": caption,        "file_name": file_name,        "row_count": len(data),        "column_count": len(data.columns)    })# Export classification-specific example datasets for convenient# placement in separate Men, Women and Unisex Typst tables.example_file_manifest = []example_table_numbers = {    "Dominant accords": "3_7",    "Dyads": "3_11",    "Triads": "3_15"}for level, table_number in example_table_numbers.items():    level_examples = next(        data for number, data, _, _ in TABLES        if number == table_number.replace("_", ".")    )    for classification in GENDERS:        subset = (            level_examples[                level_examples["Classification"] == classification            ]            .copy()            .reset_index(drop=True)        )        file_name = (            f"table_{table_number}_{typst_slug(classification)}_examples.csv"        )        subset.to_csv(            TYPST_DATA_DIR / file_name,            index=False,            encoding="utf-8"        )        example_file_manifest.append({            "analytical_level": level,            "classification": classification,            "file_name": file_name,            "row_count": len(subset),            "profile_count": subset["Accord / profile"].nunique()        })# Preserve the ranked source export separately for auditability.ranked_examples_path = TYPST_DATA_DIR / "fragrance_examples_ranked_audit.csv"examples.sort_values(    ["analytical_level", "classification", "profile", "rank_by_votes", "rating_count"],    ascending=[True, True, True, True, False]).to_csv(ranked_examples_path, index=False, encoding="utf-8")# Export the data behind the six figures as separate CSV files.figure_data = {    "figure_3_1_accord_prevalence": prevalence,    "figure_3_3_dominant_profile_segmentation": dominant,    "figure_3_4_dyad_profile_segmentation": dyads,    "figure_3_5_triad_profile_segmentation": triads,    "figure_3_6_segment_distribution": segment_summary}for file_stem, data in figure_data.items():    data.to_csv(        TYPST_DATA_DIR / f"{file_stem}.csv",        index=False,        encoding="utf-8"    )pd.DataFrame(table_manifest).to_csv(    TYPST_DATA_DIR / "table_manifest.csv",    index=False,    encoding="utf-8")pd.DataFrame(example_file_manifest).to_csv(    TYPST_DATA_DIR / "example_file_manifest.csv",    index=False,    encoding="utf-8")readme = """# Chapter 3 — Typst data exportsThis folder contains the CSV datasets used to build Chapter 3 tables and figures in Typst.## Tables- `table_manifest.csv` lists all 19 publication table datasets.- Tables 3.7, 3.11 and 3.15 contain the top-three fragrance examples for every eligible profile at the dominant-accord, dyad and triad levels.- Classification-specific example files are also supplied for Men, Women and Unisex.- Example tables omit the ranking column from publication-facing files. The separate `fragrance_examples_ranked_audit.csv` retains the source rank.## TypstAfter copying this folder into the Typst project, read a dataset with:```typst#let rows = csv("chapter_3_typst/table_3_8_dyad_segmentation_benchmarks.csv")```Use the first row as the header row. Convert numeric strings to numeric values where calculations or numeric alignment are needed.## FiguresThe figure-data CSV files contain the source values for the Chapter 3 plots. Figure image exports remain in the Notebook 19 visualisations output directory.## Ranking and tiesAll rows supplied by Notebook 18 with `rank_by_votes` from 1 through 3 are retained. If a profile has a tie at the third position, the additional tied fragrance rows are retained, provided Notebook 18's source export includes them."""(TYPST_DATA_DIR / "README_typst.md").write_text(readme, encoding="utf-8")print(f"Typst-ready exports written to: {TYPST_DATA_DIR}")print(f"Publication tables exported: {len(table_manifest)}")print(f"Classification-specific example files exported: {len(example_file_manifest)}")print(f"Ranked example audit rows: {len(examples):,}")print(f"Eligible profiles covered: {len(example_group_sizes):,}")display(pd.DataFrame(table_manifest))display(pd.DataFrame(example_file_manifest))

## 1. Tables 3.1–3.19 — publication table rendering

Run the following table batches separately. Each cell is deliberately kept below the Databricks output limit.

In [0]:

# General Chapter 3 tables — batch 1 of 5

import traceback

TABLE_LOOKUP = {
    number: (data, title, caption)
    for number, data, title, caption in TABLES
}

for number in ["3.1", "3.2", "3.3", "3.4"]:
    print(f"\n{'=' * 65}")
    print(f"Rendering Table {number}")
    print(f"{'=' * 65}")

    try:
        data, title, caption = TABLE_LOOKUP[number]
        show_table(data, number, title, caption)
        print(f"Table {number} completed successfully.")

    except Exception as error:
        print(f"ERROR while rendering Table {number}: {error}")
        traceback.print_exc()


In [0]:
# General Chapter 3 tables — batch 2 of 5
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
for number in ["3.5","3.6"]:
    data, title, caption = TABLE_LOOKUP[number]
    print(f"Table {number}: {title}")
    show_table(data, number, title, caption)

In [0]:
# General Chapter 3 tables — batch 3 of 5
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
for number in ["3.8","3.9","3.10"]:
    data, title, caption = TABLE_LOOKUP[number]
    print(f"Table {number}: {title}")
    show_table(data, number, title, caption)

In [0]:
# General Chapter 3 tables — batch 4 of 5
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
for number in ["3.12","3.13","3.14"]:
    data, title, caption = TABLE_LOOKUP[number]
    print(f"Table {number}: {title}")
    show_table(data, number, title, caption)

In [0]:
# General Chapter 3 tables — batch 5 of 5
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
for number in ["3.16","3.17","3.18","3.19"]:
    data, title, caption = TABLE_LOOKUP[number]
    print(f"Table {number}: {title}")
    show_table(data, number, title, caption)

### Table 3.7 — Dominant accords: fragrance examples

The next three cells render Men, Women and Unisex separately. Each cell contains only the four segment tables for one classification.

In [0]:
TABLE_LOOKUP = {
    number: (data, title, caption)
    for number, data, title, caption in TABLES
}

data, title, caption = TABLE_LOOKUP["3.7"]

CLASSIFICATIONS = ["Men", "Women", "Unisex"]

SEGMENTS = [
    "Standout Profiles",
    "Hidden Favourites",
    "Widely Explored",
    "Quiet Profiles"
]

for classification in CLASSIFICATIONS:
    for segment in SEGMENTS:

        subset = data[
            (data["Classification"] == classification) &
            (data["Segment"] == segment)
        ].copy()

        print("\n" + "=" * 75)
        print(f"{classification} | {segment}")
        print("=" * 75)

        if subset.empty:
            print("No matching records found.")
            continue

        # Rank profiles by the combined votes of their three examples
        accord_order = (
            subset.groupby("Accord / profile")["Votes"]
            .sum()
            .sort_values(ascending=False)
            .head(3)
            .index
        )

        for rank, accord in enumerate(accord_order, start=1):

            accord_data = (
                subset[subset["Accord / profile"] == accord]
                .sort_values("Votes", ascending=False)
                .head(3)
            )

            print(f"\n  {rank}. ACCORD: {accord}")

            for _, row in accord_data.iterrows():
                print(
                    f"     - {row['Fragrance']} | "
                    f"{row['Brand']} | "
                    f"Rating: {row['Rating']:.2f} | "
                    f"Votes: {row['Votes']:,} | "
                    f"Profile frequency: {row['Profile frequency']:,}"
                )

In [0]:
# Table series 3.7 — Men: four segment tables
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
data, title, caption = TABLE_LOOKUP["3.7"]
show_table(
    data,
    "3.7",
    title,
    caption,
    classifications=["Men"],
    segments=SEGMENTS
)

display(
    show_table(
        data,
        "3.7",
        title,
        caption,
        classifications=["Men"],
        segments=["Standout Profiles"]
    )
)

In [0]:
# Table series 3.7 — Women: four segment tables
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
data, title, caption = TABLE_LOOKUP["3.7"]
show_table(
    data,
    "3.7",
    title,
    caption,
    classifications=["Women"],
    segments=SEGMENTS
)

In [0]:
# Table series 3.7 — Unisex: four segment tables
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
data, title, caption = TABLE_LOOKUP["3.7"]
show_table(
    data,
    "3.7",
    title,
    caption,
    classifications=["Unisex"],
    segments=SEGMENTS
)

### Table 3.11 — Dyads: fragrance examples

The next three cells render Men, Women and Unisex separately. Each cell contains only the four segment tables for one classification.

In [0]:
# Table series 3.11 — Men: four segment tables
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
data, title, caption = TABLE_LOOKUP["3.11"]
show_table(
    data,
    "3.11",
    title,
    caption,
    classifications=["Men"],
    segments=SEGMENTS
)

In [0]:
# Table series 3.11 — Women: four segment tables
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
data, title, caption = TABLE_LOOKUP["3.11"]
show_table(
    data,
    "3.11",
    title,
    caption,
    classifications=["Women"],
    segments=SEGMENTS
)

In [0]:
# Table series 3.11 — Unisex: four segment tables
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
data, title, caption = TABLE_LOOKUP["3.11"]
show_table(
    data,
    "3.11",
    title,
    caption,
    classifications=["Unisex"],
    segments=SEGMENTS
)

### Table 3.15 — Triads: fragrance examples

The next three cells render Men, Women and Unisex separately. Each cell contains only the four segment tables for one classification.

In [0]:
# Table series 3.15 — Men: four segment tables
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
data, title, caption = TABLE_LOOKUP["3.15"]
show_table(
    data,
    "3.15",
    title,
    caption,
    classifications=["Men"],
    segments=SEGMENTS
)

In [0]:
# Table series 3.15 — Women: four segment tables
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
data, title, caption = TABLE_LOOKUP["3.15"]
show_table(
    data,
    "3.15",
    title,
    caption,
    classifications=["Women"],
    segments=SEGMENTS
)

In [0]:
# Table series 3.15 — Unisex: four segment tables
TABLE_LOOKUP = {number: (data, title, caption) for number, data, title, caption in TABLES}
data, title, caption = TABLE_LOOKUP["3.15"]
show_table(
    data,
    "3.15",
    title,
    caption,
    classifications=["Unisex"],
    segments=SEGMENTS
)

## 2. Figure 3.1 — Top Five Accord Prevalence by Classification

**Caption:** Comparison of the five most prevalent recorded accords in Men, Women and Unisex fragrances. Values are within-classification prevalence percentages.

In [0]:
# FIGURE 3.1 — TOP FIVE ACCORD PREVALENCE
fig_3_1, axes = plt.subplots(1,3,figsize=(17,8),dpi=180,sharex=True)
for ax, gender in zip(axes,GENDERS):
    d=prevalence[prevalence.gender_clean==gender].nlargest(5,"prevalence_pct").sort_values("prevalence_pct")
    bars=ax.barh(d.accord.str.title(),d.prevalence_pct,color=STYLE[gender],height=.62)
    for bar,v in zip(bars,d.prevalence_pct):
        ax.text(v+1,bar.get_y()+bar.get_height()/2,f"{v:.1f}%",va="center",fontsize=9)
    ax.set_title(gender,loc="left",fontfamily="DejaVu Serif",fontsize=15,fontweight="bold")
    ax.set_xlim(0,100); ax.set_xlabel("Prevalence (%)"); ax.grid(axis="x",color=STYLE["grid"]); ax.set_axisbelow(True)
    ax.tick_params(length=0,labelsize=9)
    [sp.set_visible(False) for sp in ax.spines.values()]
fig_3_1.text(.06,.97,"FRAGRANCE INTELLIGENCE  /  CHAPTER 3",color=STYLE["accent"],fontsize=9,fontweight="bold")
fig_3_1.text(.06,.93,"Top Five Accord Prevalence by Classification",fontfamily="DejaVu Serif",fontsize=21,fontweight="bold")
fig_3_1.text(.06,.885,"Percentage of fragrances within each classification containing each accord.",color=STYLE["muted"],fontsize=12)
fig_3_1.subplots_adjust(left=.08,right=.98,top=.78,bottom=.12,wspace=.35)
display(fig_3_1); plt.close(fig_3_1)

## 3. Figure 3.2 — Accord Prevalence Across Gender Classifications

**Caption:** Heatmap of within-classification prevalence for the 30 accords with the largest prevalence spread across the three classifications.

In [0]:
# FIGURE 3.2 — ACCORD PREVALENCE HEATMAP
hw=prevalence.pivot(index="accord",columns="gender_clean",values="prevalence_pct").reindex(columns=GENDERS)
hw["spread"]=hw.max(axis=1)-hw.min(axis=1)
hw=hw.nlargest(30,"spread")[GENDERS]
fig_3_2,ax=plt.subplots(figsize=(10,12),dpi=180)
cmap=LinearSegmentedColormap.from_list("ivory_bronze",[STYLE["bg"],"#D8C3AD",STYLE["accent"]])
im=ax.imshow(hw.values,aspect="auto",cmap=cmap,vmin=0,vmax=100)
ax.set_yticks(np.arange(len(hw))); ax.set_yticklabels(hw.index.str.title(),fontsize=9)
ax.set_xticks(np.arange(3)); ax.set_xticklabels(GENDERS,fontsize=10)
for i in range(len(hw)):
    for j in range(3):
        ax.text(j,i,f"{hw.iloc[i,j]:.1f}%",ha="center",va="center",fontsize=8,color=STYLE["text"])
ax.tick_params(length=0)
[sp.set_visible(False) for sp in ax.spines.values()]
cb=fig_3_2.colorbar(im,ax=ax,fraction=.025,pad=.02); cb.set_label("Prevalence (%)"); cb.outline.set_visible(False)
fig_3_2.text(.06,.97,"FRAGRANCE INTELLIGENCE  /  CHAPTER 3",color=STYLE["accent"],fontsize=9,fontweight="bold")
fig_3_2.text(.06,.93,"Accord Prevalence Across Gender Classifications",fontfamily="DejaVu Serif",fontsize=20,fontweight="bold")
fig_3_2.text(.06,.885,"Top 30 accords ranked by maximum prevalence spread.",color=STYLE["muted"],fontsize=12)
fig_3_2.subplots_adjust(left=.24,right=.91,top=.90,bottom=.06)
display(fig_3_2); plt.close(fig_3_2)
display(
    hw.reset_index()
)

## 4. Figure 3.3 — Dominant Accord Attention and Rating Segmentation

**Caption:** Eligible dominant accord profiles plotted by mean votes per fragrance and mean rating. Colour indicates the relative segment; dashed lines show the classification-specific medians.

In [0]:
# FIGURE 3.3 — DOMINANT ACCORD SEGMENTATION
FIGURE_OBJECTS_CH3 = []

def plot_level_scatter(data, title, fig_num):
    fig,axes=plt.subplots(1,3,figsize=(17,7.5),dpi=180,sharey=True)
    for ax,gender in zip(axes,GENDERS):
        d=data[data.gender_clean==gender]
        xm=d.mean_votes.median(); ym=d.mean_rating.median()
        for seg in SEGMENTS:
            p=d[d.segment.astype(str)==seg]
            ax.scatter(p.mean_votes,p.mean_rating,s=40,color=SEGMENT_COLORS[seg],
                       alpha=.78,edgecolors=STYLE["bg"],linewidths=.5,label=seg)
        ax.axvline(xm,color=STYLE["muted"],ls="--",lw=1); ax.axhline(ym,color=STYLE["muted"],ls="--",lw=1)
        ax.set_title(gender,loc="left",fontfamily="DejaVu Serif",fontsize=15,fontweight="bold")
        ax.set_xlabel("Mean votes per fragrance"); ax.grid(color=STYLE["grid"],lw=.7); ax.set_axisbelow(True)
        ax.tick_params(length=0,labelsize=9); [sp.set_visible(False) for sp in ax.spines.values()]
    axes[0].set_ylabel("Mean fragrance rating")
    handles,labels=axes[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc="lower center",bbox_to_anchor=(.5,.02),ncol=4,frameon=False,fontsize=9)
    fig.text(.06,.97,"FRAGRANCE INTELLIGENCE  /  CHAPTER 3",color=STYLE["accent"],fontsize=9,fontweight="bold")
    fig.text(.06,.93,title,fontfamily="DejaVu Serif",fontsize=20,fontweight="bold")
    fig.text(.06,.885,"Classification-specific medians define the four relative segments.",color=STYLE["muted"],fontsize=12)
    fig.subplots_adjust(left=.08,right=.98,top=.78,bottom=.20,wspace=.25)
    FIGURE_OBJECTS_CH3.append((fig, f"figure_3_{fig_num}_segmentation"))
    display(fig)

plot_level_scatter(dominant,"Dominant Accord Profiles — Attention and Rating",3)

## 5. Figure 3.4 — Dyad Attention and Rating Segmentation

**Caption:** Eligible ordered dyad profiles plotted by mean votes per fragrance and mean rating. Colour indicates the relative segment; dashed lines show the classification-specific medians.

In [0]:
plot_level_scatter(dyads,"Dyad Profiles — Attention and Rating",4)

## 6. Figure 3.5 — Triad Attention and Rating Segmentation

**Caption:** Eligible ordered triad profiles plotted by mean votes per fragrance and mean rating. Colour indicates the relative segment; dashed lines show the classification-specific medians.

In [0]:
plot_level_scatter(triads,"Triad Profiles — Attention and Rating",5)

## 7. Figure 3.6 — Cross-Level Comparison of Profile Segmentation

**Caption:** Number of eligible profiles in each median-based segment, compared across dominant accords, dyads and triads within each classification.

In [0]:
# FIGURE 3.6 — CROSS-LEVEL SEGMENT DISTRIBUTION
plot=segment_summary.copy()
fig_3_6,axes=plt.subplots(1,3,figsize=(17,8),dpi=180,sharey=True)
for ax,gender in zip(axes,GENDERS):
    d=plot[plot.gender_clean==gender]
    p=d.pivot_table(index="analytical_level",columns="segment",values="profile_count",aggfunc="sum",fill_value=0)
    p=p.reindex(index=LEVELS,columns=SEGMENTS,fill_value=0)
    bottom=np.zeros(len(p))
    for seg in SEGMENTS:
        vals=p[seg].values
        ax.bar([str(x) for x in p.index],vals,bottom=bottom,color=SEGMENT_COLORS[seg],label=seg,width=.62)
        bottom+=vals
    ax.set_title(gender,loc="left",fontfamily="DejaVu Serif",fontsize=15,fontweight="bold")
    ax.tick_params(axis="x",rotation=18,length=0,labelsize=9); ax.tick_params(axis="y",length=0)
    ax.grid(axis="y",color=STYLE["grid"]); ax.set_axisbelow(True)
    [sp.set_visible(False) for sp in ax.spines.values()]
axes[0].set_ylabel("Eligible profile count")
handles,labels=axes[0].get_legend_handles_labels()
fig_3_6.legend(handles,labels,loc="lower center",bbox_to_anchor=(.5,.02),ncol=4,frameon=False,fontsize=9)
fig_3_6.text(.06,.97,"FRAGRANCE INTELLIGENCE  /  CHAPTER 3",color=STYLE["accent"],fontsize=9,fontweight="bold")
fig_3_6.text(.06,.93,"Profile Segmentation Across Analytical Levels",fontfamily="DejaVu Serif",fontsize=20,fontweight="bold")
fig_3_6.text(.06,.885,"Stacked eligible profile counts by segment, analytical level and classification.",color=STYLE["muted"],fontsize=12)
fig_3_6.subplots_adjust(left=.08,right=.98,top=.78,bottom=.20,wspace=.28)
FIGURE_OBJECTS_CH3.append((fig_3_6,"figure_3_6_cross_level_segmentation"))
display(fig_3_6)

## 8. Export draft figures — PNG and SVG

Run after reviewing the rendered figures. This export cell saves the six figures to the shared visualisations volume. Table exports are intentionally deferred until the table design has been reviewed.

In [0]:
# Export all six Chapter 3 figures as high-resolution PNG and SVG masters.
# Run after all six figure cells have been executed.
FIGURES_CH3 = [
    (fig_3_1,"figure_3_1_top_five_accord_prevalence"),
    (fig_3_2,"figure_3_2_accord_prevalence_heatmap"),
    *FIGURE_OBJECTS_CH3
]
assert len(FIGURES_CH3) == 6, f"Expected 6 figures, found {len(FIGURES_CH3)}"
saved=[]
for fig,stem in FIGURES_CH3:
    for ext,kwargs in [("png",{"dpi":400,"bbox_inches":"tight"}),("svg",{"bbox_inches":"tight"})]:
        path=OUTPUT_DIR/f"{stem}.{ext}"
        fig.savefig(path,format=ext,facecolor=STYLE["bg"],**kwargs)
        assert path.is_file() and path.stat().st_size>0, f"Export failed: {path}"
        saved.append(path)
print("Verified figure exports:",len(saved))
for p in saved: print(p)

## 9. Review checklist- [ ] Confirm all 19 table datasets render from the current Notebook 18 exports.- [ ] Confirm the complete example datasets include every eligible profile: 95 dominant accords, 276 dyads and 104 triads. Each profile must have at least three fragrances; tied third-place records must be retained when supplied by Notebook 18.- [ ] Confirm the example exports contain at least 285, 828 and 312 rows respectively, with every exported rank in the top three vote positions and all profile groups represented.- [ ] Review all six figures for readability, label density, ordering and colour.- [ ] Run the Typst export cell and confirm all 19 table datasets, nine classification-specific example datasets, five figure-data datasets and both manifests are present in the Typst output folder.